# EDA

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import joblib
import os

### Load the dataset

In [3]:
df = pd.read_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\final_image_dataset.csv")
IMG_DIR = r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\raw_images"

#### 1: Visual sample inspection (REAL vs FAKE grid)

In [4]:
sample_df = df.sample(9, random_state=42)

fig = make_subplots(
                    rows=3,
                    cols=3,
                    subplot_titles=[row.label for row in sample_df.itertuples()]
                    )

for i, row in enumerate(sample_df.itertuples()):
    img_path = f"{IMG_DIR}/{row.image_id}.jpg"
    img = Image.open(img_path)

    r = i // 3 + 1
    c = i % 3 + 1

    fig.add_trace(
                    go.Image(z=img),
                    row=r,
                    col=c
    )

fig.update_layout(
                    height=1000,
                    width=1000,
                    title="Sample REAL vs FAKE Images"
                )

fig.show()  

joblib.dump(fig, r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\charts\eda\visual_sample_inspection.pkl")
print(" visual sample inspection saved as pkl file")

 visual sample inspection saved as pkl file


#### 2. Image Dimension Analysis (width/height spread)

In [5]:
widths, heights = [], []

for image_id in df['image_id']:
    img_path = f"{IMG_DIR}/{image_id}.jpg"
    with Image.open(img_path) as img:
        w, h = img.size
        widths.append(w)
        heights.append(h)

df['width'] = widths
df['height'] = heights
df['aspect_ratio'] = df['width'] / df['height']

print("Width  -> min:", min(widths), " max:", max(widths), " most common:", pd.Series(widths).mode()[0])
print("Height -> min:", min(heights), " max:", max(heights), " most common:", pd.Series(heights).mode()[0])
print("\nAspect ratio distribution:")
print(df['aspect_ratio'].describe())

Width  -> min: 127  max: 1080  most common: 256
Height -> min: 128  max: 2440  most common: 256

Aspect ratio distribution:
count    5088.000000
mean        0.969960
std         0.267974
min         0.442623
25%         0.701299
50%         1.000000
75%         1.000000
max         3.076923
Name: aspect_ratio, dtype: float64


#### 3: Color mode check (RGB vs Grayscale vs RGBA)

In [6]:
modes = []

for image_id in df['image_id']:
    img_path = f"{IMG_DIR}/{image_id}.jpg"
    with Image.open(img_path) as img:
        modes.append(img.mode)

df['color_mode'] = modes
print(df['color_mode'].value_counts())

color_mode
RGB    5086
L         2
Name: count, dtype: int64


##### 5: File Size Distribution Analysis

In [ ]:
file_sizes_kb = []

for image_id in df['image_id']:
    path = f"{IMG_DIR}/{image_id}.jpg"
    size_kb = os.path.getsize(path) / 1024
    file_sizes_kb.append(size_kb)

df['file_size_kb'] = file_sizes_kb
print(df['file_size_kb'].describe())

count    5088.000000
mean       86.866883
std       104.769381
min         2.380859
25%         8.787842
50%         9.895020
75%       156.722900
max       737.534180
Name: file_size_kb, dtype: float64


##### 8: Demographic × Label cross-analysis

In [ ]:
cross_tab = pd.crosstab(df['gender'], df['label'], normalize='index') * 100
print("Label % within each gender:\n", cross_tab)

cross_tab2 = pd.crosstab(df['age_group'], df['label'], normalize='index') * 100
print("\nLabel % within each age group:\n", cross_tab2)

Label % within each gender:
 label         fake        real
gender                        
female    0.000000  100.000000
male      0.000000  100.000000
unknown  95.520674    4.479326

Label % within each age group:
 label                fake        real
age_group                            
18-25            0.000000  100.000000
26-35            0.000000  100.000000
36-50            0.000000  100.000000
50+              0.000000  100.000000
not_applicable  94.579227    5.420773
under_18         0.000000  100.000000


In [ ]:
# Calculate percentage within each gender
cross_tab = pd.crosstab(
    df['gender'],
    df['label'],
    normalize='index'
) * 100

cross_tab = cross_tab.reset_index().melt(
    id_vars='gender',
    var_name='Label',
    value_name='Percentage'
)

# Create chart
fig = px.bar(
    cross_tab,
    x='gender',
    y='Percentage',
    color='Label',
    barmode='stack',
    text='Percentage',
    title='Gender-wise Distribution of REAL and FAKE Images',
    color_discrete_map={
        'REAL': "#2ECC7A",
        'FAKE': '#E74C3C'
    },
    labels={
        'gender': 'Gender',
        'Percentage': 'Percentage (%)'
    }
)

fig.update_traces(
    texttemplate='%{text:.1f}%',
    textposition='inside'
)

fig.update_layout( height = 500, width = 500,
    yaxis=dict(range=[0, 100], ticksuffix='%'),
    template='plotly_white',
    legend_title='Image Label'
)

fig.show()


joblib.dump(fig, r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\charts\eda\gender_wise_dist_real_and_fake.pkl")
print(" gender-wise distr of real and fake images saved as pkl file")


 gender-wise distr of real and fake images saved as pkl file


In [ ]:

# Calculate percentage within each age group
cross_tab2 = pd.crosstab(
    df['age_group'],
    df['label'],
    normalize='index'
) * 100

cross_tab2 = cross_tab2.reset_index().melt(
    id_vars='age_group',
    var_name='Label',
    value_name='Percentage'
)

# Create chart
fig = px.bar(
    cross_tab2,
    x='age_group',
    y='Percentage',
    color='Label',
    barmode='stack',
    text='Percentage',
    title='Age Group-wise Distribution of REAL and FAKE Images',
    color_discrete_map={
        'REAL': "#8C2CED",
        'FAKE': '#E74C3C'
    },
    labels={
        'age_group': 'Age Group',
        'Percentage': 'Percentage (%)'
    }
)

fig.update_traces(
    texttemplate='%{text:.1f}%',
    textposition='inside'
)

fig.update_layout( height = 500, width = 500,
    yaxis=dict(range=[0, 100], ticksuffix='%'),
    template='plotly_white',
    legend_title='Image Label'
)

fig.show()


joblib.dump(fig, r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\charts\eda\age_group_dist_real_and_fake.pkl")
print(" age-group distr of real and fake images saved as pkl file")


 age-group distr of real and fake images saved as pkl file


##### Save the dataset:

In [ ]:
df.to_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\eda_final_dataset.csv", index=False)
print("Saved as data/eda_final_dataset.csv")

Saved as data/eda_final_dataset.csv


### Association between variables (Cramér's V)

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import plotly.express as px

df = pd.read_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data\eda_final_dataset.csv")

def cramers_v(x, y):
    table = pd.crosstab(x, y)
    chi2, p, _, _ = chi2_contingency(table)
    n = table.values.sum()
    r, k = table.shape
    # bias-corrected Cramér's V
    phi2 = max(0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    r_c = r - (r - 1) ** 2 / (n - 1)
    k_c = k - (k - 1) ** 2 / (n - 1)
    return np.sqrt(phi2 / min(k_c - 1, r_c - 1)), p

cols = ["label", "gender", "age_group"]
v = pd.DataFrame(index=cols, columns=cols, dtype=float)
p_vals = v.copy()

for a in cols:
    for b in cols:
        v.loc[a, b], p_vals.loc[a, b] = (1.0, 0.0) if a == b else cramers_v(df[a], df[b])

fig = px.imshow(v, text_auto=".3f", color_continuous_scale="Blues", #type: ignore
                zmin=0, zmax=1, title="Association between variables")


fig.update_layout( height = 500, width = 500,
                        yaxis=dict(range=[0, 100]),
                        template='plotly_white',
                        legend_title='Image Label'
                )

fig.show()

joblib.dump(fig, r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\charts\eda\correlation_matrixs.pkl")
print(" Correlation Matrixs saved as pkl file")


print(p_vals.round(4))

# Percentage of FAKE within each group
print('-'* 50)
print(pd.crosstab(df["gender"], df["label"], normalize="index") * 100)
print('-'* 50)
print(pd.crosstab(df["age_group"], df["label"], normalize="index") * 100)


 Correlation Matrixs saved as pkl file
           label  gender  age_group
label        0.0     0.0        0.0
gender       0.0     0.0        0.0
age_group    0.0     0.0        0.0
--------------------------------------------------
label         fake        real
gender                        
female    0.000000  100.000000
male      0.000000  100.000000
unknown  95.520674    4.479326
--------------------------------------------------
label                fake        real
age_group                            
18-25            0.000000  100.000000
26-35            0.000000  100.000000
36-50            0.000000  100.000000
50+              0.000000  100.000000
not_applicable  94.579227    5.420773
under_18         0.000000  100.000000


### 3. Image features vs the label (numeric correlation):

In [ ]:
feature_cols = [
    col for col in ["width", "height", "aspect_ratio", "file_size_kb", "brightness"]
    if col in df.columns
]

num_df = df[feature_cols].copy()
num_df["label_is_fake"] = (df["label"] == "FAKE").astype(int)

corr = num_df.corr()

fig = px.imshow(
    corr.round(2),
    text_auto=True,
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Correlation of image features with the label"
)

fig.update_layout(width=600, height=520)
fig.show()

joblib.dump(fig, r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\charts\eda\correlation_image_feature_label.pkl")
print(" Correlation Matrix image feature with label saved as pkl file")


 Correlation Matrix image feature with label saved as pkl file


### 1. Label share inside each demographic group:


### Label share with gender group:

In [ ]:
for col in ["gender"]:

    table = pd.crosstab(
        df[col],
        df["label"],
        normalize="index"
    ) * 100

    fig = px.imshow(
        table.round(1),
        text_auto=True,
        color_continuous_scale="Blues",
        labels=dict(x="Label", y=col),
        title=f"Label share (%) inside each gender"
    )

    fig.update_layout(width=500, height=500)
    fig.show()

    joblib.dump(fig, r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\charts\eda\correlation_gender_group_label.pkl")
    print(" Correlation Matrix gender group label saved as pkl file")


    

 Correlation Matrix gender group label saved as pkl file


### Label share with age_group:

In [ ]:
for col in ["age_group"]:

    table = pd.crosstab(
        df[col],
        df["label"],
        normalize="index"
    ) * 100

    fig = px.imshow(
        table.round(1),
        text_auto=True,
        color_continuous_scale="Blues",
        labels=dict(x="Label", y=col),
        title=f"Label share (%) inside each age group"
    )

    fig.update_layout(width=500, height=500)
    fig.show()

    joblib.dump(fig, r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\charts\eda\correlation_age_group_label.pkl")
    print(" Correlation Matrix age_group label saved as pkl file")


    

 Correlation Matrix age_group label saved as pkl file
